# Phase 6–7 — LSTM autoencoder and consensus

The Isolation Forest (`03_isolation_forest`) is strong on point shocks but blind
to sustained level shifts: its 13-week rolling window **absorbs** a new level
(recomputed z-score 3.01 → 0.79 across an injected plateau). This notebook adds
a sequence model that sees the *shape* of the sales trajectory, evaluates it on
the same injection harness, and combines both models into a consensus.

**Key design points:**
- The AE does **not** reuse the IF's feature vector: it sees a 13-week window of
  normalized sales + `sin`/`cos` week-of-year. Feeding it z-scores would inherit
  the IF's absorption.
- Normalization is **per series and global** (median / IQR), not rolling — so a
  late level shift stays a visible offset instead of becoming the new normal.
- The latent bottleneck is tuned for **discrimination, not reconstruction**:
  latent = 4 detects far better than 8 or 16 despite a higher val loss.
- The consensus is `max(p_if, p_ae)` over percentile ranks, at a shared 5% FP budget.

Reusable code lives in `src/models/autoencoder.py`, `src/evaluation/ae_eval.py`
and `src/evaluation/consensus_eval.py`.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch

from src.config import DATA_DIR
from src.data.aggregate import get_spark
from src.evaluation.injection import build_injection_pool
from src.models.isolation_forest import load_bundle, score_rows, FEATURE_COLS
from src.models.autoencoder import (
    build_training_windows, build_series_channels, make_windows,
    LSTMAutoencoder, get_device, train_autoencoder, save_ae, load_ae,
    reconstruction_error, log_ae_run,
)
from src.evaluation.ae_eval import build_ae_eval, recall_by_type_ae
from src.evaluation.consensus_eval import (
    build_consensus_eval, recall_table, tier_distribution, log_consensus_run,
)

dev = get_device()
print("device:", dev)

device: mps


## 1. Training windows

Each dense series becomes a `(T, 3)` array `[sales_norm, sin(woy), cos(woy)]`,
then sliding windows of 13 weeks. A window scores its **centre** week. The AE
needs no YoY, so it can window over every active week, including the first year.

Windows are subsampled (50 per series, ≤ 800k total) so training fits in RAM.

In [3]:
X = build_training_windows(sample_per_series=50)
print("windows:", X.shape)
print("sales_norm mean/std:", X[:, :, 0].mean().round(3), X[:, :, 0].std().round(3))
print("sin range:", X[:, :, 1].min().round(2), X[:, :, 1].max().round(2))

windows: (774044, 13, 3)
sales_norm mean/std: 0.082 0.838
sin range: -1.0 1.0


`sales_norm` sits close to 0 with a compact spread (robust IQR scaling). The
slight positive mean is expected: retail sales are right-skewed, so
`(sales − median) / IQR` has a long positive tail.

## 2. Model

Encoder LSTM → 4-dim latent → decoder LSTM → reconstruction of the 3 channels.
Small on purpose (~14k parameters): a detector must *fail* to reconstruct what
it hasn't learned. The anomaly score is the reconstruction MSE on the **sales**
channel only — the seasonal channels are deterministic and trivially rebuilt.

In [4]:
m = LSTMAutoencoder(latent=4).to(dev)
print("out shape:", m(torch.randn(8, 13, 3, device=dev)).shape)
print("params:", sum(p.numel() for p in m.parameters()))

out shape: torch.Size([8, 13, 3])
params: 13575


## 3. Train (or load)

Training takes ~15 min on Apple Silicon (MPS). Set `TRAIN = True` to retrain;
otherwise the saved model in `models/autoencoder/` is loaded.

In [6]:
TRAIN = False

if TRAIN:
    model, best = train_autoencoder(X, epochs=80, patience=8, latent=4)
    save_ae(model, best)
model, cfg = load_ae()
print(f"latent={cfg['latent']}  best val loss={cfg['best_val']:.4f}")

latent=4  best val loss=0.0876


## 4. The bottleneck lesson: reconstruction ≠ detection

A quick synthetic check: take clean windows, inject a point spike (+5 at the
centre) or a sustained shift (+3 over the second half), and measure recall at a
5% FP threshold set on the clean windows.

Observed while tuning the latent size:

| latent | best val loss | spike recall | shift recall |
|---|---|---|---|
| 16 | 0.0008 | — (reconstructs everything) | — |
| 8 | 0.039 | 17.5% | 32.5% |
| **4** | 0.088 | **61.3%** | **78.9%** |

latent = 16 is wider than the 13 sales values: the AE
learns to copy its input, anomalies included. **Lower reconstruction error,
worse detector.** The cell below reruns the check on the saved model.

In [7]:
feats = pd.read_parquet(DATA_DIR / "features.parquet")
sample_ids = feats["id"].drop_duplicates().sample(300, random_state=0)
clean = np.concatenate([
    make_windows(build_series_channels(s)[0])[0]
    for _, s in feats[feats.id.isin(sample_ids)].sort_values("wm_yr_wk").groupby("id")
]).astype(np.float32)
L = clean.shape[1]

err_clean = reconstruction_error(model, clean, dev)
thr = np.percentile(err_clean, 95)

spike = clean.copy(); spike[:, L // 2, 0] += 5.0
shift = clean.copy(); shift[:, L // 2:, 0] += 3.0
for name, A in [("spike (+5 @ centre)", spike), ("shift (+3 @ 2nd half)", shift)]:
    e = reconstruction_error(model, A, dev)
    print(f"{name:22s} recall@5%FP = {(e >= thr).mean():.1%}")

spike (+5 @ centre)    recall@5%FP = 61.3%
shift (+3 @ 2nd half)  recall@5%FP = 78.9%


The AE is better on the sustained shift than on the point spike, as designed.
A single spike is one timestep out of 13, diluted by the window-mean error; a
shift contributes to every timestep it covers.

## 5. Evaluation on the injection harness

Same protocol as the IF: same injection pool (same series and target weeks),
same four anomaly types and severities, same clearly-normal negatives (|z| < 2),
same metric. Injections are applied to raw sales, then normalized with the
**clean** series' scaler so the anomaly shows as a deviation.

In [8]:
spark = get_spark()
features = spark.read.parquet(str(DATA_DIR / "features.parquet"))
pool_pd = build_injection_pool(features, n_series=3000).select("id", "wm_yr_wk").toPandas()
print("pool:", len(pool_pd), "series")

df_ae = build_ae_eval(feats, pool_pd, model, dev)
print("eval windows:", len(df_ae), "| positives:", int((df_ae.label == 1).sum()))
recall_by_type_ae(df_ae)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 16:10:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


pool: 3000 series
eval windows: 693020 | positives: 130547
=== AE recall @ 5% / 1% FP ===
spike        k=2.0: recall@5%=10.0%  recall@1%=2.1%
spike        k=2.5: recall@5%=12.3%  recall@1%=3.1%
spike        k=3.0: recall@5%=14.4%  recall@1%=3.9%
spike        k=3.5: recall@5%=16.5%  recall@1%=4.7%
spike        k=4.0: recall@5%=18.4%  recall@1%=5.2%
drop         k=2.0: recall@5%=5.2%  recall@1%=0.8%
drop         k=2.5: recall@5%=7.0%  recall@1%=0.8%
drop         k=3.0: recall@5%=6.7%  recall@1%=1.2%
drop         k=3.5: recall@5%=6.7%  recall@1%=1.4%
drop         k=4.0: recall@5%=8.4%  recall@1%=2.2%
contextual      : recall@5%=5.6%  recall@1%=1.0%
level_shift  k=2.0: recall@5%=18.7%  recall@1%=4.8%
level_shift  k=2.5: recall@5%=28.3%  recall@1%=8.4%
level_shift  k=3.0: recall@5%=38.3%  recall@1%=14.2%
level_shift  k=3.5: recall@5%=49.1%  recall@1%=21.4%
level_shift  k=4.0: recall@5%=59.5%  recall@1%=29.7%


**Reading:** level shifts are the AE's strength (≈ 60% at k = 4 vs the IF's
26%). It is weaker than the IF on spikes, contextual anomalies and drops — a
**specialist, not a replacement**. That complementarity is what the consensus
exploits.

In [9]:
run_id = log_ae_run(model, cfg, cfg["best_val"], df_ae)
print("logged AE run:", run_id)

/Users/alex/Projects/anomaly-detection-sc/.venv/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


logged AE run: d8f48d1cd03b475e80b2a4e4c7631913


## 6. Unified evaluation: both models on the same targets

To combine the models, each injected target needs **both** scores. For every
target the injected sales are built once, then scored by the IF (features
recomputed in pandas, mirroring the Spark pipeline) and by the AE (centred window).

In [10]:
if_model, bounds, thr_if, if_cols = load_bundle("isolation_forest")
if_thr = float(thr_if["threshold"])
valid = feats.dropna(subset=FEATURE_COLS).copy()
valid["anomaly_score"] = score_rows(valid, if_model, bounds, if_thr, FEATURE_COLS)["anomaly_score"].values

df = build_consensus_eval(feats, valid, pool_pd, (if_model, bounds, if_cols), if_thr, model, dev)
print("rows:", len(df), "| positives:", int((df.label == 1).sum()))

rows: 490627 | positives: 122624


## 7. Consensus

The two scores live on different scales, so each is converted to its
**percentile rank against clearly-normal weeks**. The consensus is
`max(p_if, p_ae)`, thresholded at the same 5% FP budget.

In [11]:
recall_table(df)

=== recall @ 5% FP  —  IF / AE / consensus ===
spike k=2.0       IF=26.7%  AE=11.0%  consensus=22.1%
spike k=2.5       IF=37.9%  AE=13.6%  consensus=30.4%
spike k=3.0       IF=54.6%  AE=15.6%  consensus=42.3%
spike k=3.5       IF=67.0%  AE=16.9%  consensus=54.5%
spike k=4.0       IF=80.6%  AE=19.1%  consensus=67.6%
drop k=2.0        IF= 5.3%  AE= 6.2%  consensus= 4.2%
drop k=2.5        IF=10.3%  AE= 8.3%  consensus= 6.2%
drop k=3.0        IF=13.2%  AE= 7.7%  consensus= 9.1%
drop k=3.5        IF=15.7%  AE= 8.1%  consensus= 9.6%
drop k=4.0        IF=19.0%  AE=10.2%  consensus=11.9%
contextual        IF=12.1%  AE= 6.6%  consensus=10.6%
level_shift k=2.0  IF=13.3%  AE=20.5%  consensus=19.0%
level_shift k=2.5  IF=16.3%  AE=30.3%  consensus=27.4%
level_shift k=3.0  IF=19.3%  AE=40.5%  consensus=37.2%
level_shift k=3.5  IF=22.6%  AE=51.2%  consensus=47.0%
level_shift k=4.0  IF=25.9%  AE=61.5%  consensus=57.2%


**Reading the table (k = 4):**

| | IF | AE | Consensus |
|---|---|---|---|
| spike | 81% | 19% | 68% |
| level_shift | 26% | 62% | 57% |

The consensus is slightly below the best model on each type. That is a
mathematical limit, not a flaw: at a **shared** 5% budget, `max(p_if, p_ae)`
spends part of the budget on both models' false positives. What it buys is
**robustness to the anomaly type**: its worst case across spikes and level
shifts is 57%, against 26% for the IF alone and 19% for the AE alone. A planner
cannot know in advance which kind of anomaly is coming.

*Alternative operating point:* the union of each model at its own 5% recovers
≈ (81%, 62%) but at ≈ 10% combined FP — twice the alert volume.

## 8. Agreement tiers

Each alert is labelled by which model fired at its own 5% threshold: **both**
(high confidence), **if_only** (likely a point shock), **ae_only** (likely a
sustained regime change). If the tiers are meaningful, injected spikes should
land in `if_only`/`both` and level shifts in `ae_only`/`both`.

In [12]:
tier_distribution(df)

tier           both  if_only  ae_only   none
anomaly_type                                
contextual       21      298      153   2156
drop             26      321      236   2943
level_shift   10339     9791    31835  51365
spike          1412     5600      590   5538


tier,both,if_only,ae_only,none
anomaly_type,,,,
contextual,21,298,153,2156
drop,26,321,236,2943
level_shift,10339,9791,31835,51365
spike,1412,5600,590,5538


Spikes land mostly in `if_only`, level shifts mostly in `ae_only`. Level shifts
also produce `if_only` flags: those are the **onset** weeks, which the IF catches
before its rolling window absorbs the new level, while the AE carries the
sustained plateau. The absorption story, made legible through the tiers.

In [13]:
run_id = log_consensus_run(df)
print("logged consensus run:", run_id)

logged consensus run: 5bfaae2d078143d387a55a18105f15dd


## Summary

- The AE is a **level-shift specialist**: ≈ 60% recall at k = 4 where the IF
  manages 26%, because its global normalization does not absorb sustained change.
- The latent size is tuned for **discrimination**: latent = 4 beats 8 and 16
  despite a higher reconstruction loss.
- The **consensus** is the most robust single detector across anomaly types
  (worst case 57% vs 26% / 19%), at the cost of a small haircut per type.
- **Agreement tiers** give each alert a confidence level and a nature hint —
  the input to the weekly triage dashboard (`/triage`).

IF, AE and consensus runs sit in the same MLflow experiment, with recall logged
by step = severity so the three curves overlay in the UI.